### TFBS overlapping variants: PWM + Bit score idea because the PWM matches are definitively not perfect
1. Identify sequence space where you want to identify TFBS: var_pos +- window (e.g. 15)
- get the information from the variant bcalm + metadata file
2. Look for TFBS using FIMO
3. Check for variant position overlapping hits 

In [1]:
import pandas as pd
import numpy as np
import yaml
import os
from collections import defaultdict
from importlib import reload
import math # check for nan
import matplotlib.pyplot as plt # plt.show()

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf
reload(hf)
# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [2]:
col_name = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class' # SNV
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'

write_unified_variant_map_again = False

In [3]:
# helpful functions
import ast

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x


### Planed process:
1. Load the bcalm information
2. Focus on the variants of cardiac_neuro_cava_random group
3. Run FIMO on the significant Variants using HOCOMOCO less redundant
4. HOCOMOCO: How many overlapping a Variant?
5. Run FIMO on the significant Variants using HOCOMOCO less redundant
6. JASPAR: How many overlapping a variant? 

In [8]:
variant_bcalm_metadata_df = pd.read_csv(config['files']['creating']['variants_metadata_bcalm_2025'], sep="\t")

In [ ]:
element_bcalm_metadata_df = pd.read_csv(config['files']['creating']['element_metadata_with_bcalm_2025_02'], low_memory=False, sep="\t")

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,bcalm_element_adjusted_p_value,bcalm_element_log_ratio_activity,bcalm_data_exists,label
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058.0,159752328.0,+,['SNV'],[234],['NC_000001.11:159752292:A:G'],['ref'],NaN,2.375354e-31,0.076298,True,GC_Mohlke
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023.0,230159293.0,+,['SNV'],[145],['NC_000001.11:230159168:C:T'],['ref'],NaN,NaN,NaN,False,GC_Mohlke
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136.0,230159406.0,+,"['SNV', 'indel']","[32, 192]","['NC_000001.11:230159168:C:T', 'NC_000001.11:2...","['ref', 'ref']",NaN,NaN,NaN,False,GC_Mohlke
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136.0,230159406.0,+,['indel'],[192],['NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCC...,['alt'],NaN,NaN,NaN,False,GC_Mohlke
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269.0,230161539.0,+,['SNV'],[120],['NC_000001.11:230161389:C:T'],['ref'],NaN,1.898862e-81,1.174486,True,GC_Mohlke
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr12:44940899-44941168|-...,TGTCCAAAAAAAGTAAAAGTCATAACAGAAATTGGATTTCAAATGG...,element,element inactive control,NaN,GRCh38,chr12,44940898.0,44941168.0,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,GC_Glut_Chengyu
80211,GC_Glut_Chengyu:Glut|chr2:212699381-212699650|...,TGTGAGTGCTATAATTGTAACCCTTTATAATTGACAGACATTAATT...,element,element inactive control,NaN,GRCh38,chr2,212699380.0,212699650.0,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,GC_Glut_Chengyu
80212,GC_Glut_Chengyu:Glut|chr10:26798829-26799098|-...,TTTAATAGGAGTACTATTGAATTTACATTTAATGTAGTTAATGATA...,element,element inactive control,NaN,GRCh38,chr10,26798828.0,26799098.0,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,GC_Glut_Chengyu
80213,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963.0,31261233.0,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,GC_Glut_Chengyu


In [ ]:
config['files']['creating']['variants_metadata_bcalm_2025']

In [4]:
config['files']['creating']['variants_bcalm_2025']

'/home/kisa/coding/80K_MPRA/bc_MPRAlm_results/results/variant_bcalm_80K_bbmap_std_mapq30_no_hashtag.tsv'

In [ ]:
variant_input_path = config['files']['creating']['variants_bcalm_2025']
variant_input_path = config['files']['creating']['variants_bcalm_2025_bwa_finest']
variant_input_path = "/home/kisa/coding/80K_MPRA/bc_MPRAlm_results/results/variant_bcalm_80K_bbmap_std_mapq30_no_hashtag_variant_map_unique_variant_id.tsv"
variant_bcalm_results = pd.read_csv(variant_input_path, sep="\t")
print(f"Number of unique variants: {variant_bcalm_results['variant_id'].nunique()}") # unique_variant_id: 38286 bbmap std mapq30 no hashtag: 38031 bwa_finest: 39021

Number of unique variants: 38286


In [43]:
variant_map_duplicated_ID = "/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/renamed_variant_region_map_mprasnakeflow_input.tsv.gz"
variant_map_duplicated_ID_df = pd.read_csv(variant_map_duplicated_ID, sep="\t")
if write_unified_variant_map_again:
    variant_map_duplicated_ID_df['unique_id'] = variant_map_duplicated_ID_df['ALT']
    variant_map_duplicated_ID_df_unified = variant_map_duplicated_ID_df.drop(columns=['ID']).copy()
    variant_map_duplicated_ID_df_unified = variant_map_duplicated_ID_df_unified.rename(columns={'unique_id': 'ID'})
    variant_map_duplicated_ID_df_unified[['ID', 'REF', 'ALT']].to_csv("/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/unique_id_renamed_variant_region_map_mprasnakeflow_input.tsv.gz", sep="\t", index=False, header=True)

In [44]:
variant_map = "/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/unique_id_renamed_variant_region_map_mprasnakeflow_input.tsv.gz"
variant_map_df = pd.read_csv(variant_map, sep="\t")
# variant_map_df =variant_map_duplicated_ID_df.copy()

In [45]:
metadata_file = pd.read_csv(config['files']['creating']['metadata_table_2025'], sep="\t", low_memory=False)

# list columns col_variant_class, col_variant_pos, col_SPDI, col_allele
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

# Apply the safe_eval function to the specified columns
for col in list_columns:
    metadata_file[col] = metadata_file[col].apply(safe_eval)


In [46]:
metadata_file['label'] = metadata_file[col_name].apply(hf.get_label)

In [47]:
# metadata_file['label'].value_counts()
metadata_file

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,label
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058.0,159752328.0,+,[SNV],[234],[NC_000001.11:159752292:A:G],[ref],NaN,GC_Mohlke
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023.0,230159293.0,+,[SNV],[145],[NC_000001.11:230159168:C:T],[ref],NaN,GC_Mohlke
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136.0,230159406.0,+,"[SNV, indel]","[32, 192]","[NC_000001.11:230159168:C:T, NC_000001.11:2301...","[ref, ref]",NaN,GC_Mohlke
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136.0,230159406.0,+,[indel],[192],[NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T],[alt],NaN,GC_Mohlke
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269.0,230161539.0,+,[SNV],[120],[NC_000001.11:230161389:C:T],[ref],NaN,GC_Mohlke
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr12:44940899-44941168|-...,TGTCCAAAAAAAGTAAAAGTCATAACAGAAATTGGATTTCAAATGG...,element,element inactive control,NaN,GRCh38,chr12,44940898.0,44941168.0,-,None,None,None,None,NaN,GC_Glut_Chengyu
80211,GC_Glut_Chengyu:Glut|chr2:212699381-212699650|...,TGTGAGTGCTATAATTGTAACCCTTTATAATTGACAGACATTAATT...,element,element inactive control,NaN,GRCh38,chr2,212699380.0,212699650.0,+,None,None,None,None,NaN,GC_Glut_Chengyu
80212,GC_Glut_Chengyu:Glut|chr10:26798829-26799098|-...,TTTAATAGGAGTACTATTGAATTTACATTTAATGTAGTTAATGATA...,element,element inactive control,NaN,GRCh38,chr10,26798828.0,26799098.0,-,None,None,None,None,NaN,GC_Glut_Chengyu
80213,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963.0,31261233.0,+,None,None,None,None,NaN,GC_Glut_Chengyu


In [48]:
variant_bcalm_results_duplicated_variant_id = variant_bcalm_results.merge(variant_map_df, left_on='variant_id', right_on='ID', how='left') # expected 38031 actual: 38397
variant_bcalm_results.merge(variant_map_df, left_on='variant_id', right_on='ID', how='left').shape[0] # expected 38031 actual: 38397 with unique ids: expected 38286 observed 38286
# bwa finest expected: 39021

38286

In [49]:
variant_bcalm_results_duplicated_variant_id.shape[0]

38286

#### Investigate if the duplicates have different values: Do they differ?
- all the values are the same for the sequences => just because different ref and alt but same variant ID
- Option A: rerun bcalm with a unique variant id (running)
- Option B: check if the sequences differ var_pos +- window

In [94]:
variant_bcalm_results_duplicated_variant_id_sorted = variant_bcalm_results_duplicated_variant_id.loc[variant_bcalm_results_duplicated_variant_id.duplicated(subset="ID")].sort_values(by='ID') # 366
print('Number of duplictes after matching ref and alt sequences: ', variant_bcalm_results_duplicated_variant_id_sorted.shape[0])
print('Number of duplictes after matching ref and alt sequences: ', variant_bcalm_results_duplicated_variant_id_sorted['ID'].nunique())

Number of duplictes after matching ref and alt sequences:  366
Number of duplictes after matching ref and alt sequences:  268


### For now drop duplicates and move on

In [50]:
# for now: drop duplicates
variant_bcalm_results_no_duplicates = variant_bcalm_results_duplicated_variant_id.drop_duplicates(subset=['ID'])
variant_bcalm_results_no_duplicates = variant_bcalm_results_duplicated_variant_id.drop_duplicates(subset=['ID'])

In [51]:
variant_map_df

,ID,REF,ALT
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
...,...,...,...
47039,C_positive_heart_CAD:ALT_rs7865618_rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618
47040,C_positive_heart_CAD:ALT_rs4977757_rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757
47041,C_positive_heart_CAD:ALT_rs1537373_rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373
47042,C_positive_heart_CAD:ALT_rs10811656_rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656


In [52]:
variant_bcalm_results

,logFC,AveExpr,t,P.Value,adj.P.Val,B,variant_id
0,1.504580,1.883764,20.091435,5.731286e-72,2.194280e-67,151.712356,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...
1,1.463167,0.952558,15.493182,3.153332e-45,6.036423e-41,90.909856,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...
2,1.472749,0.905810,13.946589,3.158700e-37,4.031132e-33,72.706154,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...
3,-1.065467,1.070354,-13.000985,8.742070e-35,8.367472e-31,67.820552,cardiac_neuro_cava_random:ALT_DISC1|ENSG000001...
4,0.881073,0.941552,12.781089,1.252137e-34,9.587865e-31,67.715565,cardiac_neuro_cava_random:ALT_ATR|ENSG00000175...
...,...,...,...,...,...,...,...
38281,0.001544,0.087147,0.022403,9.821312e-01,9.968147e-01,-6.882167,cardiac_neuro_cava_random:ALT_PPP2R2A|ENSG0000...
38282,0.000768,0.195887,0.011722,9.906503e-01,9.988577e-01,-6.882282,cardiac_neuro_cava_random:ALT_VCL|ENSG00000035...
38283,-0.000746,0.215606,-0.010951,9.912651e-01,9.989615e-01,-6.882390,cardiac_neuro_cava_random:ALT_FOXP1|ENSG000001...
38284,0.000422,0.149166,0.006386,9.949061e-01,9.993204e-01,-6.882670,cardiac_neuro_cava_random:ALT_ZNF148|ENSG00000...


In [53]:
# with unique variant id:
variant_bcalm_results_no_duplicates = variant_bcalm_results.merge(variant_map_df, left_on='variant_id', right_on='ID', how='inner')

In [54]:
variant_bcalm_results_no_duplicates # 38031 # unique ids: 38286 with bbmap: 38397 # bwa_finest
print(f'Number of variants: {variant_bcalm_results_no_duplicates.shape[0]}')

Number of variants: 38286


### Only significant variants are interesting for us

In [55]:
significant_variant_bcalm_results_no_duplicates  = variant_bcalm_results_no_duplicates.loc[variant_bcalm_results_no_duplicates['adj.P.Val'] < 0.1].copy()
significant_variant_bcalm_results_no_duplicates  = variant_bcalm_results_duplicated_variant_id.loc[variant_bcalm_results_duplicated_variant_id['adj.P.Val'] < 0.1].copy()

In [56]:
print(f"Number of significant variants: {significant_variant_bcalm_results_no_duplicates.shape[0]}") # 682 (with correct variant map (ID)) # 668 # 659 # bwa finest: 661 # 0.1: 926

Number of significant variants: 904


#### Found different number of significant Variants and wanted to check up: Check the overlap of the current significant variants with the langenberg data

In [57]:
gwas_result_langenberg = "/home/kisa/coding/80K_MPRA/collaborations/langenberg/sig_variants_80k_MPRA_unique_kircher.gwas_summary.tsv"
gwas_result_langenberg_df = pd.read_csv(gwas_result_langenberg, sep="\t")
gwas_result_langenberg_df_info = gwas_result_langenberg_df.loc[gwas_result_langenberg_df["mapped_trait"].notna()]
gwas_result_langenberg_df_info

,spdi,rsid,gwas_rsid,trait_reported,mapped_trait,mapped_trait_efo,study_id,source_gwas_pubmedid,num_reported
5,NC_000011.10:35289965:C:T,rs61882290,rs10836366,Lung function (FEV1/FVC),FEV/FVC ratio,http://www.ebi.ac.uk/efo/EFO_0004713,GCST007431,30804560,1
7,NC_000022.11:42115722:G:T,rs3985938,rs111404889||rs2004511||rs2011944||rs2743450||...,(Z)-4-hydroxytamoxifen to tamoxifen ratio in t...,body height||body mass index||cholesteryl este...,http://www.ebi.ac.uk/efo/EFO_0004339||http://w...,GCST008129||GCST009733||GCST011427||GCST900121...,29273807||31959995||32042192||32778093||352135...,14
31,NC_000002.12:224528404:C:T,rs11688390,rs11688390||rs3768884,Hemoglobin A1c levels||Smoking initiation,hemoglobin A1 measurement||smoking initiation,http://www.ebi.ac.uk/efo/EFO_0005670||http://w...,GCST90018958||GCST90243985,34594039||36477530,2
33,NC_000020.11:32130529:A:G,rs4911546,rs2093146||rs35786299||rs6119771||rs6141652||r...,Cigarettes smoked per day||Granulocyte percent...,cigarettes per day measurement||cortical surfa...,http://www.ebi.ac.uk/efo/EFO_0004698||http://w...,GCST004608||GCST008790||GCST90002394||GCST9013...,27863252||31511532||32888494||35835914||364775...,6
42,NC_000006.12:396320:C:T,rs12203592,rs12203592,Actinic keratosis||Acute myeloid leukemia or m...,"actinic keratosis||acute myeloid leukemia, mye...",http://purl.obolibrary.org/obo/MONDO_0004907||...,GCST000190||GCST000191||GCST000707||GCST000708...,18483556||20585627||21685912||23548203||257058...,50
...,...,...,...,...,...,...,...,...,...
776,NC_000011.10:9230572:G:A,rs11042228,rs12363645||rs7103472||rs72850544,Height||Neutrophil forward scatter||Neutrophil...,body height||neutrophil measurement,http://www.ebi.ac.uk/efo/EFO_0004339||http://w...,GCST90245848||GCST90281224||GCST90281227,36224396||37596262,3
785,NC_000012.12:2311210:T:C,rs4765914,rs4765913,Bipolar disorder,bipolar disorder,http://purl.obolibrary.org/obo/MONDO_0004985,GCST001241,21926972,1
786,NC_000001.11:147792137:T:C,rs12061877,rs12069680||rs12072205||rs12126043||rs201773321,Hematocrit||Hemoglobin||Hemoglobin concentrati...,erythrocyte count||hematocrit||hemoglobin meas...,http://www.ebi.ac.uk/efo/EFO_0004305||http://w...,GCST010083||GCST90002383||GCST90002403||GCST90...,32327693||32888494||34594039||35964923,5
806,NC_000015.10:63104844:G:A,rs2729833,rs2652834||rs2652838||rs2729788||rs8038648,Cholesterol to total lipids ratio in IDL||Chol...,"cholesterol:total lipids ratio, intermediate d...",http://www.ebi.ac.uk/efo/EFO_0004612||http://w...,GCST002223||GCST008745||GCST90092832||GCST9009...,24097068||31451708||35050183||35213538,6


In [58]:
metadata_file.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info', 'label'],
      dtype='object')

In [59]:
variant_bcalm_results_no_duplicates

,logFC,AveExpr,t,P.Value,adj.P.Val,B,variant_id,ID,REF,ALT
0,1.504580,1.883764,20.091435,5.731286e-72,2.194280e-67,151.712356,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...
1,1.463167,0.952558,15.493182,3.153332e-45,6.036423e-41,90.909856,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...
2,1.472749,0.905810,13.946589,3.158700e-37,4.031132e-33,72.706154,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:REF_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...
3,-1.065467,1.070354,-13.000985,8.742070e-35,8.367472e-31,67.820552,cardiac_neuro_cava_random:ALT_DISC1|ENSG000001...,cardiac_neuro_cava_random:ALT_DISC1|ENSG000001...,cardiac_neuro_cava_random:REF_DISC1|ENSG000001...,cardiac_neuro_cava_random:ALT_DISC1|ENSG000001...
4,0.881073,0.941552,12.781089,1.252137e-34,9.587865e-31,67.715565,cardiac_neuro_cava_random:ALT_ATR|ENSG00000175...,cardiac_neuro_cava_random:ALT_ATR|ENSG00000175...,cardiac_neuro_cava_random:REF_ATR|ENSG00000175...,cardiac_neuro_cava_random:ALT_ATR|ENSG00000175...
...,...,...,...,...,...,...,...,...,...,...
38281,0.001544,0.087147,0.022403,9.821312e-01,9.968147e-01,-6.882167,cardiac_neuro_cava_random:ALT_PPP2R2A|ENSG0000...,cardiac_neuro_cava_random:ALT_PPP2R2A|ENSG0000...,cardiac_neuro_cava_random:REF_PPP2R2A|ENSG0000...,cardiac_neuro_cava_random:ALT_PPP2R2A|ENSG0000...
38282,0.000768,0.195887,0.011722,9.906503e-01,9.988577e-01,-6.882282,cardiac_neuro_cava_random:ALT_VCL|ENSG00000035...,cardiac_neuro_cava_random:ALT_VCL|ENSG00000035...,cardiac_neuro_cava_random:REF_VCL|ENSG00000035...,cardiac_neuro_cava_random:ALT_VCL|ENSG00000035...
38283,-0.000746,0.215606,-0.010951,9.912651e-01,9.989615e-01,-6.882390,cardiac_neuro_cava_random:ALT_FOXP1|ENSG000001...,cardiac_neuro_cava_random:ALT_FOXP1|ENSG000001...,cardiac_neuro_cava_random:REF_FOXP1|ENSG000001...,cardiac_neuro_cava_random:ALT_FOXP1|ENSG000001...
38284,0.000422,0.149166,0.006386,9.949061e-01,9.993204e-01,-6.882670,cardiac_neuro_cava_random:ALT_ZNF148|ENSG00000...,cardiac_neuro_cava_random:ALT_ZNF148|ENSG00000...,cardiac_neuro_cava_random:REF_ZNF148|ENSG00000...,cardiac_neuro_cava_random:ALT_ZNF148|ENSG00000...


In [22]:
# variants which contain a ";" because they are duplicates are not split in the metadata file => not important variants at the moment
# set(variant_bcalm_results_no_duplicates['ALT'].to_list()) - set(variant_bcalm_results_no_duplicates_metadata['ALT'].to_list())

# we have too many headers with ";" so we cannot automatically explode all
# metadata_file.loc[metadata_file['name'].str.contains('GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791547A>G|SHH')].name.to_list()


In [60]:
significant_variant_bcalm_results_no_duplicates

,logFC,AveExpr,t,P.Value,adj.P.Val,B,variant_id,ID,REF,ALT
0,1.504580,1.883764,20.091435,5.731286e-72,2.194280e-67,151.712356,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...
1,1.463167,0.952558,15.493182,3.153332e-45,6.036423e-41,90.909856,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...
2,1.472749,0.905810,13.946589,3.158700e-37,4.031132e-33,72.706154,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:REF_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...
3,-1.065467,1.070354,-13.000985,8.742070e-35,8.367472e-31,67.820552,cardiac_neuro_cava_random:ALT_DISC1|ENSG000001...,cardiac_neuro_cava_random:ALT_DISC1|ENSG000001...,cardiac_neuro_cava_random:REF_DISC1|ENSG000001...,cardiac_neuro_cava_random:ALT_DISC1|ENSG000001...
4,0.881073,0.941552,12.781089,1.252137e-34,9.587865e-31,67.715565,cardiac_neuro_cava_random:ALT_ATR|ENSG00000175...,cardiac_neuro_cava_random:ALT_ATR|ENSG00000175...,cardiac_neuro_cava_random:REF_ATR|ENSG00000175...,cardiac_neuro_cava_random:ALT_ATR|ENSG00000175...
...,...,...,...,...,...,...,...,...,...,...
1023,0.204768,0.094548,3.103941,1.966206e-03,8.888302e-02,-2.134711,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,cardiac_neuro_cava_random:REF_RAD51B|ENSG00000...,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...
1025,0.243626,0.633009,3.056904,2.315348e-03,9.904232e-02,-2.138109,cardiac_neuro_cava_random:ALT_LINC01033|ENSG00...,cardiac_neuro_cava_random:ALT_LINC01033|ENSG00...,cardiac_neuro_cava_random:REF_LINC01033|ENSG00...,cardiac_neuro_cava_random:ALT_LINC01033|ENSG00...
1026,0.225364,-0.022232,3.068244,2.227046e-03,9.610763e-02,-2.143569,cardiac_neuro_cava_random:ALT_FH|ENSG000000914...,cardiac_neuro_cava_random:ALT_FH|ENSG000000914...,cardiac_neuro_cava_random:REF_FH|ENSG000000914...,cardiac_neuro_cava_random:ALT_FH|ENSG000000914...
1028,0.235757,-0.092345,3.075589,2.170350e-03,9.434085e-02,-2.146969,cardiac_neuro_cava_random:ALT_STAMBP|ENSG00000...,cardiac_neuro_cava_random:ALT_STAMBP|ENSG00000...,cardiac_neuro_cava_random:REF_STAMBP|ENSG00000...,cardiac_neuro_cava_random:ALT_STAMBP|ENSG00000...


In [61]:
# variant_bcalm_results_no_duplicates_metadata = variant_bcalm_results_no_duplicates.merge(metadata_file[['name', 'sequence', 'chr',
    #    'start', 'end', 'strand', 'variant_pos', 'SPDI', 'label']], left_on="ALT", right_on=col_name)
variant_bcalm_results_no_duplicates_metadata = significant_variant_bcalm_results_no_duplicates.merge(metadata_file[['name', 'sequence', 'chr',
       'start', 'end', 'strand', 'variant_pos', 'SPDI', 'label']], left_on="ALT", right_on=col_name)
variant_bcalm_results_no_duplicates_metadata['SPDI_elem'] = variant_bcalm_results_no_duplicates_metadata['SPDI'].apply(lambda spdi: spdi[0])


In [62]:
sig_variant_bcalm_results_no_duplicates_metadata = variant_bcalm_results_no_duplicates_metadata.loc[variant_bcalm_results_no_duplicates_metadata['adj.P.Val'] < 0.1]
sig_variant_bcalm_results_no_duplicates_metadata # 657 bwa_finest: 644, 1
sig_variant_bcalm_results_no_duplicates_metadata.label.value_counts()

# cardiac_neuro_cava_random    654
# GC_Mendelian_variants          2
# GC_Selvarajan                  1

# older data with correct variant map
# label
# cardiac_neuro_cava_random    671
# GC_Mendelian_variants          8
# GC_Selvarajan                  1
# Name: count, dtype: int64

# bwa finest with correct variant map
# label
# cardiac_neuro_cava_random    644
# GC_Selvarajan                  1

label
cardiac_neuro_cava_random    896
GC_Mendelian_variants          4
GC_Selvarajan                  1
GC_Atrial_fib                  1
Name: count, dtype: int64

In [65]:
sig_variant_bcalm_results_no_duplicates_metadata
print(sig_variant_bcalm_results_no_duplicates_metadata.shape[0])
print(gwas_result_langenberg_df_info.shape[0])
gwas_result_langenberg_df_info.merge(sig_variant_bcalm_results_no_duplicates_metadata, left_on="spdi", right_on="SPDI_elem", how="inner").shape[0] # 31/63 / 25/63 27/63
gwas_result_langenberg_df_info.merge(variant_bcalm_results_no_duplicates_metadata, left_on="spdi", right_on="SPDI_elem", how="inner").shape[0] # 45/63 / 35/63 / 27/63

902
63


27

In [66]:
significant_variants_with_langenberg = sig_variant_bcalm_results_no_duplicates_metadata.merge(gwas_result_langenberg_df_info, left_on="SPDI_elem", right_on="spdi", how="left") # 657 # 25 reported

In [26]:
31/657
63/815

0.07730061349693251

In [67]:
significant_variants_with_langenberg['abs_logFC'] = significant_variants_with_langenberg['logFC'].abs()

In [ ]:
import gzip # zipped files
import pandas as pd
from Bio import SeqIO

def read_zipped_fasta(fasta_file):
    """
    Read zipped fasta file with Biopython.
    """
    handle = gzip.open(fasta_file, 'rt')
    fasta_sequences = SeqIO.parse(handle,'fasta')
    return fasta_sequences


def fasta_to_dataframe(fasta_file, columns=[]):
    """
    Convert a fasta file to a pandas dataframe.
    """
    # case for ziped files:
    if fasta_file.endswith('.gz'):
        fasta_sequences = read_zipped_fasta(fasta_file)
    else:
        fasta_sequences = SeqIO.parse(open(fasta_file),'fasta')
    header = []
    sequence = []
    for fasta in fasta_sequences:
        header.append(fasta.id)
        sequence.append(str(fasta.seq))
    df = pd.DataFrame({'header': header, 'sequence': sequence})
    if columns != [] and len(columns) == 2:
        df.columns = columns
    return df


def write_fasta(sequence_df, output_path, header=['header', 'sequence']):
    """
    Write the fasta file with the header and sequence
    """
    with open(output_path, 'w') as f:
        for index, row in sequence_df.iterrows():
            f.write('>' + row[header[0]] + '\n' + row[header[1]] + '\n')
    return True


# prepare the strand sensitive design file:
# read the design => remove adapters => write fasta file again
design_with_adapter_path = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_removed_spaces_deduplicated_sequences_renamed.fa"
design_with_adapter_fa_df = hf.fasta_to_dataframe(design_with_adapter_path)
# remove adapter:
design_with_adapter_fa_df['sequence_without_adapter'] = design_with_adapter_fa_df['sequence'].apply(lambda seq: seq[15:-15])
design_with_adapter_fa_df['sequence_without_adapter'].str.len().value_counts()
no_adapter_output_path = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_removed_spaces_deduplicated_sequences_renamed_no_adapter.fa"
hf.write_fasta(design_with_adapter_fa_df, output_path=no_adapter_output_path, header=['header', 'sequence_without_adapter'])
# => start the workflow

True

##### Get the numbers for the variant groups
- fisher exact test top and bottom effect size: Odds Ratio: 0.23828125, P-value: 0.9711438686443264
- ranked: manwhitney u between absolute effect sizes of gwas hits and not gwas hits
    - Mann-Whitney U statistic: 7931.0, P-value: 0.4869298273688484

In [68]:
import math
# top and bottom x%
top_bottom_fraction = 0.1
sorted_sig_variant_bcalm_results_no_duplicates_metadata = significant_variants_with_langenberg.sort_values(by='logFC', ascending=False)
top_pos_effect_size_vars = sorted_sig_variant_bcalm_results_no_duplicates_metadata.head(math.floor(sig_variant_bcalm_results_no_duplicates_metadata.shape[0]*0.1))
top_neg_effect_size_vars = sorted_sig_variant_bcalm_results_no_duplicates_metadata.tail(math.floor(sig_variant_bcalm_results_no_duplicates_metadata.shape[0]*0.1))

In [ ]:
top_neg_effect_size_vars.gwas_rsid.notna().sum() # 4 0.1: 7
top_pos_effect_size_vars.gwas_rsid.notna().sum() # 1 0.1: 1

7

In [71]:
from scipy.stats import fisher_exact

# Define the observed counts
top_pos_gwas = top_pos_effect_size_vars.gwas_rsid.notna().sum()   # GWAS-reported in the top effect size group
top_pos_non_gwas = top_pos_effect_size_vars.gwas_rsid.isna().sum()   # Non-GWAS in the top effect size group
top_neg_gwas = top_neg_effect_size_vars.gwas_rsid.notna().sum()   # GWAS-reported in the bottom effect size group
top_neg_non_gwas = top_neg_effect_size_vars.gwas_rsid.isna().sum()   # Non-GWAS in the bottom effect size group


# Construct the contingency table
contingency_table = [[top_pos_gwas, top_pos_non_gwas],
                     [top_neg_gwas, top_neg_non_gwas]]

# Perform Fisher's exact test
odds_ratio, p_value = fisher_exact(contingency_table, alternative='greater')

print(f"Odds Ratio: {odds_ratio}, P-value: {p_value}")


Odds Ratio: 0.1332263242375602, P-value: 0.9966791877411328


In [72]:
sig_gwas_variants = sorted_sig_variant_bcalm_results_no_duplicates_metadata.loc[sorted_sig_variant_bcalm_results_no_duplicates_metadata.gwas_rsid.notna()]
sig_non_gwas_variants = sorted_sig_variant_bcalm_results_no_duplicates_metadata.loc[sorted_sig_variant_bcalm_results_no_duplicates_metadata.gwas_rsid.isna()]

In [73]:
abs_effect_sizes_gwas = sig_gwas_variants['abs_logFC']
abs_effect_sizes_non_gwas = sig_non_gwas_variants['abs_logFC']

In [74]:
from scipy.stats import mannwhitneyu

# Assume abs_effect_sizes_gwas and abs_effect_sizes_non_gwas are lists
U_stat, p_value = mannwhitneyu(abs_effect_sizes_gwas, abs_effect_sizes_non_gwas, alternative='greater')

print(f"Mann-Whitney U statistic: {U_stat}, P-value: {p_value}")

Mann-Whitney U statistic: 12783.0, P-value: 0.2334600751942807


#### Found interesting result: The old data (without outlier removal) had more significant variants
- What is correct?: missing variants would mean: we found false positives before (114)
  - identify the variants with the highest difference in p-value 

In [27]:
# read old variant file:
# bcalm_no_outlier_removal = "/home/kisa/coding/80K_MPRA/bc_MPRAlm_results/results/bcalm_test_toptable_bc_mpralm_80K_bbmap_std_mapq35.tsv" # 38362, 667
# bcalm_no_outlier_removal = "/home/kisa/coding/80K_MPRA/bc_MPRAlm_results/results/bcalm_test_toptable_bc_mpralm_80K_bbmap_std_mapq10.tsv" # 682
bcalm_no_outlier_removal = "/home/kisa/coding/80K_MPRA/server_results/bc_MPRAlm/bbmap_std_mapq35/toptable_bbmap_std_mapq35_NoLength_unique_variant_map_NGN2.tsv" # 821
# bcalm_no_outlier_removal = "/home/kisa/coding/80K_MPRA/server_results/bc_MPRAlm/toptable_bbmap_standard_mapq35_no_length_NGN2.tsv" # 38553, 816
bcalm_no_outlier_removal = "/home/kisa/coding/80K_MPRA/bc_MPRAlm_results/results/variant_bcalm_80K_bbmap_std_mapq30_no_hashtag_no_outlier_removal_variant_map_unique_variant_id.tsv" # 38395, 651
bcalm_no_outlier_removal_df = pd.read_csv(bcalm_no_outlier_removal, sep="\t")
print('Number of variants: ', bcalm_no_outlier_removal_df.shape[0]) # for no outlier removal with adapter on sequences and bbmap mapq30: 38395 -> 651 significant variants
bcalm_no_outlier_removal_duplicated_df = bcalm_no_outlier_removal_df.merge(variant_map_duplicated_ID_df, left_on="variant_id", right_on="ID")
bcalm_no_outlier_removal_duplicated_df = bcalm_no_outlier_removal_df.merge(variant_map_df, left_on="variant_id", right_on="ID")
sig_bcalm_no_outlier_removal_duplicated = bcalm_no_outlier_removal_duplicated_df.loc[bcalm_no_outlier_removal_duplicated_df['adj.P.Val'] < 0.05]
sig_bcalm_no_outlier_removal_duplicated['variant_id'].nunique() # 651

Number of variants:  38395


651

In [28]:
len(sig_variants_outlier_removed) # 659
len(sig_varaints_no_outlier_removed) # 651

NameError: name 'sig_variants_outlier_removed' is not defined

In [ ]:
sig_varaints_no_outlier_removed = set(sig_bcalm_no_outlier_removal_duplicated['ALT'].to_list())
all_variants_no_outlier_removal = set(bcalm_no_outlier_removal_duplicated_df["ALT"].to_list())

In [ ]:
sig_variants_outlier_removed = set(significant_variant_bcalm_results_no_duplicates['ALT'].to_list())
all_variants_outlier_removed = set(variant_bcalm_results_no_duplicates['ALT'].to_list())

In [ ]:
# set 1, set 2, significant set 1, significant set 2:
# Interesting: what is significant in set 1 but not in set 2 (Are all of these in the set 2?)

# 1. Only significant in either with or without outlier removal
only_sig_outlier_removed = sig_variants_outlier_removed - sig_varaints_no_outlier_removed
only_sig_no_outlier_removed = sig_varaints_no_outlier_removed - sig_variants_outlier_removed
print("Number of significant variants ONLY within outlier removed: ", len(only_sig_outlier_removed))
print("Number of variants ONLY within NOT outlier removed: ", len(only_sig_no_outlier_removed))

# 2. Only significant and existing in either with or without outlier removal
only_existing_with_outlier_removal = only_sig_outlier_removed - all_variants_no_outlier_removal
only_existing_no_outlier_removal = only_sig_no_outlier_removed - all_variants_outlier_removed
print("Number of significant variants from the outlier removed set which are not among the variants of the not outlier removal set: ",len(only_existing_with_outlier_removal))
print("Number of significant variants from the NO outlier removed set which are not among the variants of the set WITH outlier removal: ",len(only_existing_no_outlier_removal))


print("Intersection of both variant sets: ", len(sig_variants_outlier_removed.intersection(sig_varaints_no_outlier_removed)))

Number of significant variants ONLY within outlier removed:  43
Number of variants ONLY within NOT outlier removed:  35
Number of significant variants from the outlier removed set which are not among the variants of the not outlier removal set:  0
Number of significant variants from the NO outlier removed set which are not among the variants of the set WITH outlier removal:  0
Intersection of both variant sets:  616


#### Investigate the lost variants in the data
- find data with the highest difference

In [ ]:
bcalm_no_outlier_removal_duplicated_df.head()
bcalm_no_outlier_removal_duplicated_df.shape[0]
bcalm_no_outlier_removal_duplicated_df["variant_id"].nunique()

38395

In [ ]:
variant_bcalm_results_no_duplicates.shape[0]
variant_bcalm_results_no_duplicates["ALT"].nunique()

38286

In [ ]:
variant_bcalm_results_no_duplicates.columns
variant_bcalm_results_no_duplicates

Index(['logFC', 'AveExpr', 't', 'P.Value', 'adj.P.Val', 'B', 'variant_id',
       'ID', 'REF', 'ALT'],
      dtype='object')

In [ ]:
bcalm_no_outlier_removal_duplicated_df

In [ ]:
only_sig_no_outlier_removed

{'cardiac_neuro_cava_random:ALT_ABL1|ENSG00000097007.20|EH38E2731753_fwd_tile1-1_ABL1|ENSG00000097007.20|EH38E2731753|9-130765390-T-A',
 'cardiac_neuro_cava_random:ALT_ATM|ENSG00000149311.20|EH38E2983000_fwd_tile1-1_ATM|ENSG00000149311.20|EH38E2983000|11-108277232-T-C',
 'cardiac_neuro_cava_random:ALT_AUTS2|ENSG00000158321.19|EH38E2561987_fwd_tile1-1_AUTS2|ENSG00000158321.19|EH38E2561987|7-70575017-C-T',
 'cardiac_neuro_cava_random:ALT_BRCA2|ENSG00000139618.18|EH38E3059491_fwd_tile1-1_BRCA2|ENSG00000139618.18|EH38E3059491|13-32266296-G-A',
 'cardiac_neuro_cava_random:ALT_CACNA1C|ENSG00000151067.23|EH38E1587201_fwd_tile1-1_CACNA1C|ENSG00000151067.23|EH38E1587201|12-2547369-T-C',
 'cardiac_neuro_cava_random:ALT_CYP2C19|ENSG00000165841.11|EH38E2917481_fwd_tile1-1_CYP2C19|ENSG00000165841.11|EH38E2917481|10-94800683-G-A',
 'cardiac_neuro_cava_random:ALT_DENND5A|ENSG00000184014.9|EH38E1518980_rev_tile1-1_DENND5A|ENSG00000184014.9|EH38E1518980|11-9139654-T-C',
 'cardiac_neuro_cava_random:ALT_

### Get the ref and alt sequences

In [ ]:
metadata_file.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info', 'label'],
      dtype='object')

In [ ]:
# subset of columns from metadata_file
columns_of_interest_metadata_reference = [col_name, col_sequence]
columns_of_interest_metadata_alternative = [col_name, col_sequence, col_variant_pos, col_strand]

# left join metadata_file with variant_results_bcalm_unique_variants (REF)
variant_results_bcalm_unique_variants_sequences = significant_variant_bcalm_results_no_duplicates.merge(metadata_file[columns_of_interest_metadata_reference], left_on='REF', right_on=col_name, how='left')

# rename sequence to REF_sequence
variant_results_bcalm_unique_variants_sequences = variant_results_bcalm_unique_variants_sequences.rename(columns={col_sequence: 'REF_sequence'})
variant_results_bcalm_unique_variants_sequences = variant_results_bcalm_unique_variants_sequences.drop(columns=[col_name])

# left join metadata_file with variant_results_bcalm_unique_variants (ALT)
variant_results_bcalm_unique_variants_sequences = variant_results_bcalm_unique_variants_sequences.merge(metadata_file[columns_of_interest_metadata_alternative], left_on='ALT', right_on=col_name, how='left')
print(f"Number of na in ALT_sequence: {variant_results_bcalm_unique_variants_sequences[col_sequence].isna().sum()}")

# rename sequence to ALT_sequence
variant_results_bcalm_unique_variants_sequences = variant_results_bcalm_unique_variants_sequences.rename(columns={col_sequence: 'ALT_sequence'})
variant_results_bcalm_unique_variants_sequences = variant_results_bcalm_unique_variants_sequences.drop(columns=[col_name])

variant_results_bcalm_unique_variants_sequences

Number of na in ALT_sequence: 0


,logFC,AveExpr,t,P.Value,adj.P.Val,B,variant_id,ID,REF,ALT,REF_sequence,ALT_sequence,variant_pos,strand
0,1.504051,1.884027,20.088120,5.770173e-72,2.194455e-67,151.701840,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,TTACTAGAATGCTAGACTGAATTCTTTTTAAAAAAATTGACAGTAA...,TTACTAGAATGCTAGACTGAATTCTTTTTAAAAAAATTGACAGTAA...,[164],+
1,1.462369,0.952810,15.484087,3.392386e-45,6.450791e-41,90.824527,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,cardiac_neuro_cava_random:REF_TRIO|ENSG0000003...,cardiac_neuro_cava_random:ALT_TRIO|ENSG0000003...,ATTAACTGTAGATTGCATGCTGAATTCAGCCTAAAAACAAGTTACA...,ATTAACTGTAGATTGCATGCTGAATTCAGCCTAAAAACAAGTTACA...,[189],+
2,1.472634,0.905787,13.936694,3.401995e-37,4.312709e-33,72.618165,cardiac_neuro_cava_random:ANKZF1|ENSG000001635...,cardiac_neuro_cava_random:ANKZF1|ENSG000001635...,cardiac_neuro_cava_random:REF_ANKZF1|ENSG00000...,cardiac_neuro_cava_random:ALT_ANKZF1|ENSG00000...,CAGGTTCACTGTTGGGCTCTGATCCCACCTTCCCACCATGGGGACA...,CAGGTTCACTGTTGGGCTCTGATCCCACCTTCCCACCATGGGGACA...,[63],+
3,-1.065587,1.070453,-12.998698,8.869708e-35,8.433097e-31,67.799041,cardiac_neuro_cava_random:DISC1|ENSG0000016294...,cardiac_neuro_cava_random:DISC1|ENSG0000016294...,cardiac_neuro_cava_random:REF_DISC1|ENSG000001...,cardiac_neuro_cava_random:ALT_DISC1|ENSG000001...,AAGCTTGTGATCCTGAGAAATTTTGTGTTTTCTGTGTTGTCTATAA...,AAGCTTGTGATCCTGAGAAATTTTGTGTTTTCTGTGTTGTCTATAA...,[240],+
4,0.878341,0.940398,12.664867,4.439653e-34,3.376889e-30,66.470608,cardiac_neuro_cava_random:ATR|ENSG00000175054....,cardiac_neuro_cava_random:ATR|ENSG00000175054....,cardiac_neuro_cava_random:REF_ATR|ENSG00000175...,cardiac_neuro_cava_random:ALT_ATR|ENSG00000175...,TCATTTTCTTCTCGAAAATCTTTTTCCTTAGACGAGGGACTTGACC...,TCATTTTCTTCTCGAAAATCTTTTTCCTTAGACGAGGGACTTGACC...,[52],-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663,0.336632,0.196142,3.352226,8.582533e-04,4.893588e-02,-1.077744,cardiac_neuro_cava_random:CHD7|ENSG00000171316...,cardiac_neuro_cava_random:CHD7|ENSG00000171316...,cardiac_neuro_cava_random:REF_CHD7|ENSG0000017...,cardiac_neuro_cava_random:ALT_CHD7|ENSG0000017...,CCACAATTTTAGAACCACTGGTATGAAGGAAAGAGTACAGGCTTTG...,CCACAATTTTAGAACCACTGGTATGAAGGAAAGAGTACAGGCTTTG...,[142],+
664,-0.279172,0.228403,-3.373760,7.817764e-04,4.553100e-02,-1.128655,cardiac_neuro_cava_random:FOXP2|ENSG0000012857...,cardiac_neuro_cava_random:FOXP2|ENSG0000012857...,cardiac_neuro_cava_random:REF_FOXP2|ENSG000001...,cardiac_neuro_cava_random:ALT_FOXP2|ENSG000001...,ATGGTGCTAGTGCATTCAGTTTGTTTACTTTGGTTTCTATGTCTGC...,ATGGTGCTAGTGCATTCAGTTTGTTTACTTTGGTTTCTATGTCTGC...,[237],+
665,0.282491,0.089217,3.359834,8.225243e-04,4.732439e-02,-1.141670,cardiac_neuro_cava_random:H1-3|ENSG00000124575...,cardiac_neuro_cava_random:H1-3|ENSG00000124575...,cardiac_neuro_cava_random:REF_H1-3|ENSG0000012...,cardiac_neuro_cava_random:ALT_H1-3|ENSG0000012...,CATGGGTTTGTGAGTTAGTTTCTTACAGGTAACTAAAGTCTCTCCC...,CATGGGTTTGTGAGTTAGTTTCTTACAGGTAACTAAAGTCTCTCCC...,[147],-
666,-0.307265,-0.113889,-3.356373,8.355329e-04,4.800023e-02,-1.153930,cardiac_neuro_cava_random:DMD|ENSG00000198947....,cardiac_neuro_cava_random:DMD|ENSG00000198947....,cardiac_neuro_cava_random:REF_DMD|ENSG00000198...,cardiac_neuro_cava_random:ALT_DMD|ENSG00000198...,TTTTAAGGACCCCTGTGATTAAATCCTGCCCACTCAGATAATCTGG...,TTTTAAGGACCCCTGTGATTAAATCCTGCCCACTCAGATAATCTGG...,[98],-


In [ ]:
variant_results_bcalm_unique_variants_sequences['label'] = variant_results_bcalm_unique_variants_sequences['ID'].apply(hf.get_label)
variant_results_bcalm_unique_variants_sequences['label'].value_counts()

label
cardiac_neuro_cava_random    664
GC_Mendelian_variants          3
GC_Selvarajan                  1
Name: count, dtype: int64

### Get sequence space for fimo
- found out that the variant position is not defined based on the string but based on the strand they are on

In [ ]:
def find_differences(seq1, seq2):
    return [i for i, (a, b) in enumerate(zip(seq1, seq2)) if a != b]

def validate_variant_pos(row, col_ref_seq='REF_sequence', col_alt_seq='ALT_sequence', col_var_pos='variant_pos', col_strand='strand', seq_length=270):
    """Compares Ref and alt sequences and checks if the variant position is the difference between the two sequences"""
    if len(row[col_var_pos]) > 1:
        raise ValueError("Expected only one element in the variant pos column")
    variant_pos = row[col_var_pos][0]
    if row[col_strand] == "-":
        variant_pos = seq_length - (variant_pos + 1) # (0-based result)

    variants = find_differences(row[col_ref_seq], row[col_alt_seq])
    if len(variants) > 1:
        print(f"WARNING: only one difference expected {len(variants)} detected")
        print(row['ALT'])
    if variant_pos == variants[0]:
        return True
    else:
        print(row['ALT'])
        print(row[col_var_pos])
    return False

In [ ]:
variant_results_bcalm_unique_variants_sequences

- GC_Mendelian_variants:ALT_chr7:156791255G>C|SHH_chr7:156791274T>TTAAGGAAGTGATT|SHH 
- seems to be not correctly associated to the reference (76 differences detected but only 1 expected)
- wrong variant position for regions on the - strand

In [ ]:
variant_results_bcalm_unique_variants_sequences['valid_variant_pos'] = variant_results_bcalm_unique_variants_sequences.apply(validate_variant_pos, axis=1)
variant_results_bcalm_unique_variants_sequences['valid_variant_pos'].sum() # 371
# variant_results_bcalm_unique_variants_sequences.shape[0] # 668

GC_Mendelian_variants:ALT_chr7:156791255G>C|SHH_chr7:156791274T>TTAAGGAAGTGATT|SHH
GC_Mendelian_variants:ALT_chr7:156791255G>C|SHH_chr7:156791274T>TTAAGGAAGTGATT|SHH
[154]


667

In [ ]:
# TODO: collect reference and alternative sequences => fasta only window around variant position
# TODO: remove ":" from header => problems in FIMO

In [ ]:
variant_results_bcalm_unique_variants_sequences

In [ ]:
#### FIMO commands:
motif_set = config['files']['creating']['local_h12core_tf_less_redundant_motif_set']
motif_set = '/data/cephfs-2/unmirrored/groups/kircher/Kaikkonen_2023/filesFromMinnaKaikkonen/github_repo/STARseqCNN/JASPAR2022_CORE_vertebrates_non-redundant_pfms_meme_nice.txt'
#! remember changing motif set name
motif_set_name = 'H12Core'
motif_set_name = 'JASPAR2022'
neuro_variant_output = config['files']['creating']['fimo_neuro_variant_output_dir']
fimo_output_name = "fimo_neuro_variants_window_size_8" # +-8
fimo_output_name = "fimo_neuro_variants_JASPAR2022" # +-10 not -10 and +9
fimo_output_name = "fimo_neuro_variants" # +-10 not -10 and +9
fimo_output_name = "fimo_open_in_neuro_variants_H12Core"
fimo_output_name = "fimo_open_in_neuro_variants_JASPAR2022"
fimo_output_path = os.path.join(neuro_variant_output, fimo_output_name)
neuro_fimo_output_table = os.path.join(config['files']['creating']['fimo_neuro_variant_output_dir'], f'neuro_significant_fimo_sequences_{combined_fimo_table.shape[0]}_seqs.fasta')

# # example: would require to install meme in current environment
# # !conda activate meme;
# !fimo --max-stored-scores 1000000 --o {fimo_output_path} {motif_set} {neuro_fimo_output_table};

In [ ]:
fimo_result = "/home/kisa/coding/80K_MPRA/fimo_data/variant_tf_search/fimo_cava_variants_without_adapters_JASPAR/fimo.tsv"

In [ ]:
fimo_result
fimo_result_df = pd.read_csv(fimo_result, sep="\t", comment='#')

In [ ]:
fimo_result_df

,motif_id,motif_alt_id,sequence_name,start,stop,strand,score,p-value,q-value,matched_sequence
0,MA1594.1_ZNF382,ZNF382,REF_RAD51B|ENSG00000182185.19|EH38E1723158_fwd...,144,167,-,23.5657,6.960000e-09,0.000464,AGATGGCATTACAACTGATCCCAC
1,MA1594.1_ZNF382,ZNF382,ALT_RAD51B|ENSG00000182185.19|EH38E1723158_fwd...,144,167,-,23.5657,6.960000e-09,0.000464,AGATGGCATTACAACTGATCCCAC
2,MA1642.1_NEUROG2,NEUROG2,REF_MYBPC3|ENSG00000134571.12|EH38E2957377_rev...,104,116,+,16.1220,8.550000e-09,0.000591,GGGACAGATGGCC
3,MA1642.1_NEUROG2,NEUROG2,ALT_MYBPC3|ENSG00000134571.12|EH38E2957377_rev...,104,116,+,16.1220,8.550000e-09,0.000591,GGGACAGATGGCC
4,MA0149.1_EWSR1-FLI1,EWSR1-FLI1,REF_LMNA|ENSG00000160789.24|EH38E1387656_fwd_t...,162,179,-,18.8553,1.020000e-08,0.000678,GGGAGGAGGGAAGCAAGG
...,...,...,...,...,...,...,...,...,...,...
21103,MA0162.4_EGR1,EGR1,REF_SFPQ|ENSG00000116560.12|EH38E2802412_rev_t...,150,163,-,10.0163,1.000000e-04,0.551000,GCCCTCCCCCACTG
21104,MA0162.4_EGR1,EGR1,ALT_SFPQ|ENSG00000116560.12|EH38E2802412_rev_t...,150,163,-,10.0163,1.000000e-04,0.551000,GCCCTCCCCCACTG
21105,MA0834.1_ATF7,ATF7,REF_ATR|ENSG00000175054.16|EH38E3544406_rev_ti...,193,206,+,6.4000,1.000000e-04,0.870000,AAGTGAAGTCATAG
21106,MA0834.1_ATF7,ATF7,ALT_ATR|ENSG00000175054.16|EH38E3544406_rev_ti...,193,206,+,6.4000,1.000000e-04,0.870000,AAGTGAAGTCATAG


### Read the motifs from the meme file

In [ ]:
def read_meme_motifs(file_path):
    motifs = []
    with open(file_path, 'r') as file:
        lines = file.readlines()
        motif = None
        for line in lines:
            if line.startswith('MOTIF'):
                if motif:
                    motifs.append(motif)
                motif = {'name': line.strip().split()[1], 'matrix': []}
            elif line[0].isdigit():
                motif['matrix'].append([float(x) for x in line.strip().split()])
        if motif:
            motifs.append(motif)
    return motifs

In [ ]:
meme_h12_core = "/home/kisa/coding/80K_MPRA/fimo_data/H12CORE_less_redundant.meme"
motifs_h12_core_list = read_meme_motifs(meme_h12_core)

In [ ]:
from Bio.motifs import meme

with open(meme_h12_core) as f_in:
    h12_core_motifs = meme.read(f_in)

for motif in h12_core_motifs:
    for instance in motif.instances:
        print(instance.motif_name, instance.sequence_name, instance.strand, instance.pvalue)

ValueError: Improper MEME XML input file. XML root tag should start with <MEME version= ...